### Import Libraries

In [4]:
import pandas as pd 
import requests
from sklearn.preprocessing import MinMaxScaler 
import pyodbc
from sqlalchemy import create_engine

# Extract 

In [6]:
df=pd.read_csv('technova_sales_data.csv')
df

,TransactionID,TransactionDate,Product,Category,Quantity,Price,TotalRevenue,Currency,CustomerID,CustomerName,PaymentMethod,Region
0,1,2024-10-05,Laptop,Electronics,5,1200,6000,USD,104,Dana White,PayPal,East
1,2,2024-04-22,Laptop,Electronics,5,1200,6000,USD,102,Bob Smith,Credit Card,East
2,3,2024-08-06,Smartphone,Electronics,1,800,800,GBP,103,Charlie Davis,Credit Card,South
3,4,2024-02-25,Tablet,Electronics,5,500,2500,INR,101,Alice Johnson,PayPal,North
4,5,2024-10-27,Headphones,Accessories,1,150,150,INR,101,Alice Johnson,Cryptocurrency,North
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2025-01-24,Headphones,Accessories,5,150,750,GBP,101,Alice Johnson,PayPal,East
9996,9997,2024-12-18,Tablet,Electronics,2,500,1000,USD,104,Dana White,Cryptocurrency,South
9997,9998,2024-12-18,Tablet,Electronics,4,500,2000,EUR,101,Alice Johnson,PayPal,North
9998,9999,2024-07-20,Smartwatch,Wearables,4,250,1000,GBP,104,Dana White,Bank Transfer,West


In [7]:
#Define API URL 
api_url ='https://api.exchangerate-api.com/v4/latest/USD'

In [8]:
# Make api request 
response = requests.get(api_url)
response

<Response [200]>

In [9]:
# convert response to json
if response.status_code == 200:
    data=response.json()
print(response.json())

{'provider': 'https://www.exchangerate-api.com', 'WARNING_UPGRADE_TO_V6': 'https://www.exchangerate-api.com/docs/free', 'terms': 'https://www.exchangerate-api.com/terms', 'base': 'USD', 'date': '2025-02-24', 'time_last_updated': 1740355201, 'rates': {'USD': 1, 'AED': 3.67, 'AFN': 73.71, 'ALL': 94.5, 'AMD': 394.89, 'ANG': 1.79, 'AOA': 919.87, 'ARS': 1060.88, 'AUD': 1.57, 'AWG': 1.79, 'AZN': 1.7, 'BAM': 1.87, 'BBD': 2, 'BDT': 121.47, 'BGN': 1.87, 'BHD': 0.376, 'BIF': 2966.27, 'BMD': 1, 'BND': 1.34, 'BOB': 6.92, 'BRL': 5.71, 'BSD': 1, 'BTN': 86.7, 'BWP': 13.79, 'BYN': 3.27, 'BZD': 2, 'CAD': 1.42, 'CDF': 2855.5, 'CHF': 0.898, 'CLP': 943.36, 'CNY': 7.25, 'COP': 4094.65, 'CRC': 505.69, 'CUP': 24, 'CVE': 105.27, 'CZK': 23.95, 'DJF': 177.72, 'DKK': 7.12, 'DOP': 62.24, 'DZD': 134.81, 'EGP': 50.59, 'ERN': 15, 'ETB': 126.85, 'EUR': 0.955, 'FJD': 2.29, 'FKP': 0.79, 'FOK': 7.12, 'GBP': 0.79, 'GEL': 2.8, 'GGP': 0.79, 'GHS': 15.53, 'GIP': 0.79, 'GMD': 72.58, 'GNF': 8586.17, 'GTQ': 7.72, 'GYD': 209.23

In [10]:
#extract exchange rates
exchange_rates = data['rates']

# Transform

In [12]:
def currency_exchange(price,currency):
    if currency in exchange_rates:      # Check if the currency exists in the API data
        return price/exchange_rates[currency]    # Convert to USD
    return none                         # If currency not found, return None

In [13]:
df['Price_USD']=df.apply(lambda row:currency_exchange(row['Price'],row['Currency']),axis=1)

In [14]:
df['Price_USD']=df['Price_USD'].round(2)

#### Handling Missing Data

In [16]:
df.isnull().sum()   #there are no null values

TransactionID      0
TransactionDate    0
Product            0
Category           0
Quantity           0
Price              0
TotalRevenue       0
Currency           0
CustomerID         0
CustomerName       0
PaymentMethod      0
Region             0
Price_USD          0
dtype: int64

#### Data Normalization 

In [18]:
scaler=MinMaxScaler()
df['Price_USD']=scaler.fit_transform(df[['Price_USD']])

#### Data Duplicates

In [20]:
df.drop_duplicates()

,TransactionID,TransactionDate,Product,Category,Quantity,Price,TotalRevenue,Currency,CustomerID,CustomerName,PaymentMethod,Region,Price_USD
0,1,2024-10-05,Laptop,Electronics,5,1200,6000,USD,104,Dana White,PayPal,East,0.789840
1,2,2024-04-22,Laptop,Electronics,5,1200,6000,USD,102,Bob Smith,Credit Card,East,0.789840
2,3,2024-08-06,Smartphone,Electronics,1,800,800,GBP,103,Charlie Davis,Credit Card,South,0.666414
3,4,2024-02-25,Tablet,Electronics,5,500,2500,INR,101,Alice Johnson,PayPal,North,0.003044
4,5,2024-10-27,Headphones,Accessories,1,150,150,INR,101,Alice Johnson,Cryptocurrency,North,0.000382
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2025-01-24,Headphones,Accessories,5,150,750,GBP,101,Alice Johnson,PayPal,East,0.124335
9996,9997,2024-12-18,Tablet,Electronics,2,500,1000,USD,104,Dana White,Cryptocurrency,South,0.328658
9997,9998,2024-12-18,Tablet,Electronics,4,500,2000,EUR,101,Alice Johnson,PayPal,North,0.344180
9998,9999,2024-07-20,Smartwatch,Wearables,4,250,1000,GBP,104,Dana White,Bank Transfer,West,0.207736


#### Data Formatting

In [22]:
df['TransactionDate']=pd.to_datetime(df['TransactionDate'],format='%Y-%m-%d')

#### Create New Column for Total Value

In [24]:
df['Total_Value']=df['Price']*df['Quantity']
df

,TransactionID,TransactionDate,Product,Category,Quantity,Price,TotalRevenue,Currency,CustomerID,CustomerName,PaymentMethod,Region,Price_USD,Total_Value
0,1,2024-10-05,Laptop,Electronics,5,1200,6000,USD,104,Dana White,PayPal,East,0.789840,6000
1,2,2024-04-22,Laptop,Electronics,5,1200,6000,USD,102,Bob Smith,Credit Card,East,0.789840,6000
2,3,2024-08-06,Smartphone,Electronics,1,800,800,GBP,103,Charlie Davis,Credit Card,South,0.666414,800
3,4,2024-02-25,Tablet,Electronics,5,500,2500,INR,101,Alice Johnson,PayPal,North,0.003044,2500
4,5,2024-10-27,Headphones,Accessories,1,150,150,INR,101,Alice Johnson,Cryptocurrency,North,0.000382,150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2025-01-24,Headphones,Accessories,5,150,750,GBP,101,Alice Johnson,PayPal,East,0.124335,750
9996,9997,2024-12-18,Tablet,Electronics,2,500,1000,USD,104,Dana White,Cryptocurrency,South,0.328658,1000
9997,9998,2024-12-18,Tablet,Electronics,4,500,2000,EUR,101,Alice Johnson,PayPal,North,0.344180,2000
9998,9999,2024-07-20,Smartwatch,Wearables,4,250,1000,GBP,104,Dana White,Bank Transfer,West,0.207736,1000


#### Categorical Encoding 

In [26]:
# Get the count of each unique category in the 'Product_Type' column
print(df['Product'].value_counts())

Product
Smartwatch    1454
Tablet        1449
Monitor       1444
Smartphone    1417
Headphones    1417
Laptop        1416
Keyboard      1403
Name: count, dtype: int64


In [27]:
df['Category_Code'] = df['Product'].map({'Smartwatch': 1, 'Tablet': 2, 'Monitor': 3,'Smartphone':4,'Headphones':5,'Laptop':6,'Keyboard':7})
df

,TransactionID,TransactionDate,Product,Category,Quantity,Price,TotalRevenue,Currency,CustomerID,CustomerName,PaymentMethod,Region,Price_USD,Total_Value,Category_Code
0,1,2024-10-05,Laptop,Electronics,5,1200,6000,USD,104,Dana White,PayPal,East,0.789840,6000,6
1,2,2024-04-22,Laptop,Electronics,5,1200,6000,USD,102,Bob Smith,Credit Card,East,0.789840,6000,6
2,3,2024-08-06,Smartphone,Electronics,1,800,800,GBP,103,Charlie Davis,Credit Card,South,0.666414,800,4
3,4,2024-02-25,Tablet,Electronics,5,500,2500,INR,101,Alice Johnson,PayPal,North,0.003044,2500,2
4,5,2024-10-27,Headphones,Accessories,1,150,150,INR,101,Alice Johnson,Cryptocurrency,North,0.000382,150,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2025-01-24,Headphones,Accessories,5,150,750,GBP,101,Alice Johnson,PayPal,East,0.124335,750,5
9996,9997,2024-12-18,Tablet,Electronics,2,500,1000,USD,104,Dana White,Cryptocurrency,South,0.328658,1000,2
9997,9998,2024-12-18,Tablet,Electronics,4,500,2000,EUR,101,Alice Johnson,PayPal,North,0.344180,2000,2
9998,9999,2024-07-20,Smartwatch,Wearables,4,250,1000,GBP,104,Dana White,Bank Transfer,West,0.207736,1000,1


In [28]:
pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


#### Converting dataframe to CSV file

In [30]:
df.to_csv('transformed_data.csv', index=False)

In [69]:
transform_df=pd.read_csv('transformed_data.csv')

AttributeError: 'Index' object has no attribute 'sum'

In [71]:
len(transform_df)

10000

#### SQL Connection 

sql connection format -
engine = create_engine("mssql+pyodbc://SERVER_NAME/DATABASE_NAME?driver=ODBC+Driver+17+for+SQL+Server")

In [65]:
# Create an engine for SQL Server connection
engine = create_engine("mssql+pyodbc://DESKTOP-CABV4JI\\SQLEXPRESS/TechNovaDB?driver=ODBC+Driver+17+for+SQL+Server")

#### Insert the Dataset to SQL Server

When you execute SQL queries in SQL Server (using SSMS), you're working directly with the database for analysis. The Python-to-SQL Server connection, on the other hand, is used for the Load step of the ETL process. Here's a quick breakdown:

Python Connection:

Purpose: Automates the process of loading (inserting) your transformed data from a CSV (or another source) into the SQL Server database.
Usage: You write a Python script that reads your data, connects to SQL Server, and then inserts that data into a table.
SQL Queries in SSMS:

Purpose: Once your data is loaded into the database, you use SSMS to execute SQL queries for analysis, reporting, and visualization.
Usage: You interact directly with the data already stored in SQL Server to generate insights.
So, the Python connection is part of your ETL pipeline (specifically the "Load" part), while executing SQL queries in SSMS is about analyzing and using the loaded data.

In [83]:
# Insert the dataset into SQL Server in bulk
df.to_sql("SalesData",engine,if_exists="replace",index=False)
print("Data successfully inserted into SQL Server!")

Data successfully inserted into SQL Server!


#### Automate Data Extraction, Transformation, and Loading (ETL) with Python
Python Script (ETL Process): We'll write a Python script that extracts data from an API, transforms it (like currency conversion), and loads it into SQL Server.

In [8]:
# Step 1: Extract data from API
url = "https://api.exchangerate-api.com/v4/latest/USD"
response = requests.get(url)
data=response.json()

In [12]:
# Extract exchange rates from the JSON response
rates=data['rates']
df_new=pd.DataFrame(data)